# Machine Learning Forecasting

## Objective

This notebook evaluates whether machine learning models can improve forecasting performance beyond traditional statistical forecasting methods.

Previous notebooks showed that demand is primarily driven by strong yearly seasonality and long-term growth. Holt-Winters Exponential Smoothing achieved strong performance and serves as the statistical benchmark for the machine learning models tested here.

The focus of this notebook is to:

- engineer lag, rolling, calendar, and business features
- train machine learning models using a temporal validation split
- detect and remove target leakage
- compare model performance against the statistical benchmark
- interpret feature importance from a business forecasting perspective


## 1. Setup

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File paths
from pathlib import Path

# Modeling
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

## 2. Data Preparation

The machine learning models use the forecasting base dataset at the original product-market-month grain. This preserves product, market, lifecycle, supply, inventory, and demand signal information that was not available in the aggregated statistical forecasting notebook.


In [ ]:
project_root = Path.cwd().parent
data_path = project_root / "data" / "processed" / "vw_forecasting_base.csv"

# SSMS exports may not include headers, so column names are defined explicitly.
columns = [
    "month",
    "product_id",
    "market_id",
    "category",
    "brand",
    "lifecycle_stage",
    "region",
    "lead_time_months",
    "strategic_priority",
    "actual_demand_units",
    "promo_flag",
    "campaign_intensity",
    "demand_signal_quality",
    "available_supply_units",
    "production_capacity_units",
    "supply_constraint_flag",
    "actual_shipped_units",
    "unfulfilled_demand_units",
    "service_level_pct",
    "ending_inventory_units",
    "inventory_risk_status"
]

df = pd.read_csv(
    data_path,
    header=None,
    names=columns,
    sep=";",
    keep_default_na=False
)

df.head()

In [ ]:
rows, cols = df.shape
print(f"Rows: {rows:,} | Columns: {cols}")

In [ ]:
df.describe()

In [ ]:
# Convert month to datetime for chronological sorting and temporal splitting.
df["month"] = pd.to_datetime(
    df["month"],
    format="%d/%m/%Y",
    dayfirst=True
)

df.info()

## 3. Feature Engineering

The feature engineering strategy translates the demand patterns found during EDA into model-ready predictors:

- calendar features capture seasonality
- lag features capture historical demand behavior
- rolling features capture recent trend and local demand level
- rolling standard deviations capture volatility

All lag and rolling features are created within each product-market time series to avoid mixing demand histories across different business entities.


In [ ]:
# Sort each product-market time series chronologically before creating lag features.
df = (
    df.sort_values(["product_id", "market_id", "month"])
      .reset_index(drop=True)
)

df.head()

### Calendar Features

In [ ]:
df["year"] = df["month"].dt.year
df["month_number"] = df["month"].dt.month
df["quarter"] = df["month"].dt.quarter

### Lag Features

In [ ]:
# Lag features use previous demand from the same product-market time series.
for lag in [1, 3, 6, 12]:
    df[f"lag_{lag}"] = (
        df.groupby(["product_id", "market_id"])["actual_demand_units"]
          .shift(lag)
    )

### Rolling Mean Features

In [ ]:
# Shift before rolling to prevent using the current month's demand to predict itself.
df["rolling_mean_3"] = (
    df.groupby(["product_id", "market_id"])["actual_demand_units"]
      .transform(lambda x: x.shift(1).rolling(3).mean())
)

df["rolling_mean_6"] = (
    df.groupby(["product_id", "market_id"])["actual_demand_units"]
      .transform(lambda x: x.shift(1).rolling(6).mean())
)

### Rolling Volatility Features

In [ ]:
df["rolling_std_3"] = (
    df.groupby(["product_id", "market_id"])["actual_demand_units"]
      .transform(lambda x: x.shift(1).rolling(3).std())
)

df["rolling_std_6"] = (
    df.groupby(["product_id", "market_id"])["actual_demand_units"]
      .transform(lambda x: x.shift(1).rolling(6).std())
)

## 4. Feature Validation

Lag and rolling features naturally introduce null values at the beginning of each product-market time series. These rows are removed before modeling because not all historical features are available for them.


In [ ]:
engineered_features = [
    "lag_1",
    "lag_3",
    "lag_6",
    "lag_12",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_std_3",
    "rolling_std_6"
]

df[engineered_features].isna().sum()

In [ ]:
df_model = df.dropna(subset=engineered_features).copy()

print(f"Original shape: {df.shape}")
print(f"Modeling shape: {df_model.shape}")

In [ ]:
df_model[
    [
        "month",
        "product_id",
        "market_id",
        "actual_demand_units",
        "lag_1",
        "lag_12",
        "rolling_mean_3",
        "rolling_std_3"
    ]
].head(15)

## 5. Categorical Feature Encoding

Categorical variables are one-hot encoded so they can be used by tree-based machine learning models. This is appropriate here because the dataset is relatively small and the categorical cardinality is manageable.


In [ ]:
categorical_cols = [
    "product_id",
    "market_id",
    "category",
    "brand",
    "lifecycle_stage",
    "region",
    "demand_signal_quality",
    "inventory_risk_status"
]

df_model[categorical_cols].nunique()

In [ ]:
df_model_encoded = pd.get_dummies(
    df_model,
    columns=categorical_cols,
    drop_first=True
)

print(f"Before encoding: {df_model.shape}")
print(f"After encoding:  {df_model_encoded.shape}")

## 6. Modeling Dataset and Leakage Prevention

The target variable is `actual_demand_units`.

Some columns must be excluded because they are either identifiers/date fields or future outcome variables. In particular, `actual_shipped_units`, `unfulfilled_demand_units`, and `service_level_pct` are not known at forecast creation time and would introduce target leakage.


In [ ]:
target = "actual_demand_units"

cols_to_exclude = [
    "actual_demand_units",
    "month",

    # Leakage features: future operational outcomes, not valid predictors.
    "actual_shipped_units",
    "unfulfilled_demand_units",
    "service_level_pct"
]

feature_cols = [
    col
    for col in df_model_encoded.columns
    if col not in cols_to_exclude
]

X = df_model_encoded[feature_cols]
y = df_model_encoded[target]

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 7. Temporal Train/Test Split

A random train/test split would leak future information into training. Instead, the data is split chronologically:

- Training period: all available observations before 2025
- Test period: 2025 observations

This simulates the real forecasting scenario: using historical information to predict future demand.


In [ ]:
train = df_model_encoded[
    df_model_encoded["month"] < "2025-01-01"
]

test = df_model_encoded[
    df_model_encoded["month"] >= "2025-01-01"
]

X_train = train[feature_cols]
y_train = train[target]

X_test = test[feature_cols]
y_test = test[target]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows:  {len(X_test):,}")

## 8. Random Forest Regressor

Random Forest is used as the first machine learning benchmark. It can capture nonlinear relationships and feature interactions, but it does not naturally extrapolate trend as directly as classical time series models.


In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_predictions = rf.predict(X_test)

In [ ]:
rf_mae = mean_absolute_error(
    y_true=y_test,
    y_pred=rf_predictions
)

rf_mse = mean_squared_error(
    y_true=y_test,
    y_pred=rf_predictions
)

rf_mape = mean_absolute_percentage_error(
    y_true=y_test,
    y_pred=rf_predictions
)

print(f"RF MAE:  {rf_mae:,.2f}")
print(f"RF MSE:  {rf_mse:,.2f}")
print(f"RF MAPE: {rf_mape:.4f}")

In [ ]:
rf_feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

rf_feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10, 8))

sns.barplot(
    data=rf_feature_importance.head(15),
    x="importance",
    y="feature"
)

plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### Random Forest Interpretation

After removing leakage variables, Random Forest performance is substantially worse than the Holt-Winters benchmark. Feature importance shows that `lag_12` dominates the model, reinforcing the finding that yearly seasonality is the strongest predictor of future demand.

This indicates that the model is learning the same seasonal structure identified during EDA, but it is not modeling the overall trend as effectively as Holt-Winters.


## 9. XGBoost Regressor

XGBoost is evaluated as a stronger tree-based model. It can capture nonlinear patterns and interactions more effectively than Random Forest, making it a useful final machine learning challenger against the statistical forecasting benchmark.


In [ ]:
xgb = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

xgb_predictions = xgb.predict(X_test)

In [ ]:
xgb_mae = mean_absolute_error(
    y_true=y_test,
    y_pred=xgb_predictions
)

xgb_mse = mean_squared_error(
    y_true=y_test,
    y_pred=xgb_predictions
)

xgb_mape = mean_absolute_percentage_error(
    y_true=y_test,
    y_pred=xgb_predictions
)

print(f"XGB MAE:  {xgb_mae:,.2f}")
print(f"XGB MSE:  {xgb_mse:,.2f}")
print(f"XGB MAPE: {xgb_mape:.4f}")

In [ ]:
xgb_feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": xgb.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

xgb_feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10, 8))

sns.barplot(
    data=xgb_feature_importance.head(15),
    x="importance",
    y="feature"
)

plt.title("Top 15 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### XGBoost Interpretation

XGBoost improves on Random Forest but still does not outperform the best statistical forecasting model. Its feature importance again identifies `lag_12` as the dominant predictor, confirming that the core demand signal is primarily seasonal.

The model uses additional features such as recent demand, promotions, production capacity, campaign intensity, and category indicators, but these do not provide enough incremental value to beat Holt-Winters.


## 10. Final Model Comparison

In [ ]:
model_comparison = pd.DataFrame({
    "model": [
        "Naive Forecast",
        "Seasonal Naive",
        "XGBoost",
        "Random Forest",
        "Seasonal Naive + Trend Adjustment",
        "Holt-Winters Exponential Smoothing"
    ],
    "mape": [
        0.5897,
        0.1155,
        xgb_mape,
        rf_mape,
        0.0203,
        0.0139
    ]
})

model_comparison["mape_pct"] = model_comparison["mape"] * 100
model_comparison = model_comparison.sort_values("mape")

model_comparison

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=model_comparison,
    x="mape_pct",
    y="model"
)

plt.title("Model Comparison by MAPE")
plt.xlabel("MAPE (%)")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

# Conclusions

This notebook evaluated whether machine learning models could improve forecasting performance beyond the statistical models developed previously.

The results show that machine learning did not outperform the best statistical forecasting approach. Holt-Winters Exponential Smoothing remained the strongest model, achieving a MAPE of 1.39%, compared with 11.58% for XGBoost and 12.71% for Random Forest.

The feature importance analysis provides the key explanation: both machine learning models relied heavily on `lag_12`, confirming that yearly seasonality is the dominant predictive signal. This aligns with the exploratory analysis and the strong performance of seasonal statistical models.

A target leakage issue was also identified and corrected. Initial Random Forest results appeared unrealistically strong because future outcome variables such as shipped units and unfulfilled demand were included as predictors. Removing those variables produced a realistic modeling setup and a more credible comparison.

The main conclusion is that this demand forecasting problem is highly structured and primarily driven by trend and seasonality. In this context, a simpler and more interpretable statistical model outperforms more complex machine learning approaches.

From a business perspective, Holt-Winters is the recommended forecasting model because it provides the best accuracy while remaining transparent, explainable, and operationally appropriate.
